# AI MEGADADOS 25-2

**NOME**: Ricardo Luz Carvalho

## Observações
- Esta prova é formada por 7 (sete) questões e tem **duração de 2h00**.
- Os exercícios com correção automática não terão correção manual, mas poderá haver checks de qualidade/honestidade. Caso tente burlar os testes, sua nota de prova será zero e será enviado ao regime disciplinar. O que é burlar? Por exemplo, para a pergunta "calcule a média da coluna x", o correto é fazer "SELECT AVG(x) FROM tabela", já fazer "SELECT 100.10" é considerado burlar os testes, pois sua query não funciona de forma genérica, apenas cola o próprio valor de resultado.
- Você pode trazer uma folha de papel (tamanho próximo a A4 / oficio) preenchida dos dois lados e utilizar como consulta. A folha pode ser tanto impressa quanto escrita a mão, mas precisa ser física (não pode acessar no computador). Não é permitido utilizar lupa rsrs.
- Será permitido utilizar o Workbench para testar queries e fazer forward / reverse engineering. Feche as abas com respostas realizadas durante o semestre.
- Não é permitido compartilhar qualquer material durante a prova.
- Não serão permitidas consultas a outros meios (material blackboard, sites, notebooks das aulas, github) impressos ou digitais, nem contactar outras pessoas. Caso ocorra, a nota da prova será zero e será enviado ao regime disciplinar.
- Não será permitido o uso de ferramentas de IA (copilot, chatgpt e afins). Você precisará providenciar um ambiente seguro para a prova, sem extensões instaladas (não basta desativar) nem sites abertos. Caso ocorra, a nota da prova será zero e será enviado ao regime disciplinar.

## Parte 1 - Plataforma de Podcast XTREMO!

Na parte 1 da prova de Megadados, iremos trabalhar com a base de dados **XTREMO**.

O banco de dados `xtremo` é um sistema relacional sintético projetado para **gerenciar informações de um ecossistema de podcasts**. O foco é manter o registro de aspectos da produção, distribuição e monetização de conteúdo de áudio. Ele contém tabelas que cobrem as seguintes áreas principais:

- **Conteúdo:** A tabela **`podcast`** armazena os dados principais dos programas, enquanto a tabela **`episodio`** detalha cada episódio.
- **Participantes:** O banco gerencia informações sobre os **`hosts`** (apresentadores) e **`convidados`**, incluindo detalhes de contato e participação. A tabela **`podcast_host`** relaciona hosts a podcasts, e **`episodio_convidado`** liga convidados a episódios.
- **Monetização:** As tabelas **`anunciante`**, **`tipo_anuncio`**, e **`anuncio`** lidam com dados comerciais. A tabela **`anuncio_episodio`** associa anúncios a episódios específicos, registrando informações como valor pago e ouvintes no momento da veiculação.
- **Distribuição e Engajamento:** As tabelas **`plataforma_distribuicao`** e **`episodio_plataforma`** controlam a presença dos episódios em diferentes plataformas. A tabela **`topico`** e **`episodio_topico`** permitem categorizar o conteúdo e as **`mensagens`** coletam dados de interação do público.

### Insper autograding!

Para entrega dos exercício na parte 1, iremos utilizar o `insper autograding`. Garanta que a biblioteca está instalada em seu ambiente Python.

**Sugestão**: crie uma pasta para a prova e copie o `.env` utilizado nas aulas para esta pasta.

### Instalação da base

Execute os scripts `xtremo_0001.sql` e `xtremo_0002.sql` no **MySQL Workbench**. Estes scripts criam um banco `xtremo` e inserem dados de exemplo para resolução da prova.

O banco de dados pode ser representado pelo seguinte diagrama do modelo relacional (também disponível em PDF na pasta `docs`):

<img src="img/xtremo.png">

## Como resolver os exercícios?

Crie o banco da prova em sua máquina (passo anterior). Utilize o MySQL Workbench ou o conector para testar as *queries*. Quando estiver bastante certo de que a resposta está correta, faça a submissão para o servidor.

Em alguns momentos, pode ser necessário analisar a resposta esperada do servidor para entender modificações na sua solução, mesmo que não seja algo explícito no enunciado.

## Import das bibliotecas

Vamos realizar o import das bibliotecas.

In [1]:
import mysql.connector
from functools import partial
import os
import insperautograder.jupyter as ia
from dotenv import load_dotenv

E vamos criar nosso HELPER de conexão com o banco! Perceba que, uma vez configurado o `.env` não precisaremos mais informar usuários, senhas e URLs!

In [7]:
load_dotenv(override=True)

def get_connection_helper():

    def run_db_query(connection, query, args=None):
        with connection.cursor() as cursor:
            print("Executando query:")
            for result in cursor.execute(query, multi=True):
                if result.with_rows:
                    for row in result.fetchall():
                        print(row)
                else:
                    print(f"{result.rowcount} linhas afetadas.")

    connection = mysql.connector.connect(
        host=os.getenv("MD_DB_SERVER"),
        user=os.getenv("MD_DB_USERNAME"),
        password=os.getenv("MD_DB_PASSWORD"),
        database="xtremo",
    )
    return connection, partial(run_db_query, connection)


connection, db = get_connection_helper()

### Notas

Para conferir a nota da correção automática da prova, utilize:

In [3]:
ia.grades(task="ai_md_25_2")

| Atividade   | Exercício   | Peso   | Nota   | Nota Sem Atraso   | Nota Com Atraso   |
|-------------|-------------|--------|--------|-------------------|-------------------|

In [4]:
ia.grades(by="TASK", task="ai_md_25_2")

| Tarefa   | Nota   | Conta como ATV?   |
|----------|--------|-------------------|

**Obs**:

- As questões 1 a 6 valem **7.0 pontos** na nota da prova.
    - Estas questões serão corrigidas **apenas** pela correção automática. Não haverá nota manual, mas haverá check de honestidade, conforme regras da prova.
    - Confira os pesos pela API, a média é ponderada.
    - No servidor, a nota da atividade estará no intervalo 0 a 10. Multiplique por `0.7` para saber a real contribuição na nota da prova.
- A questão 7 vale **3.0 pontos** na nota da prova.

Logo, a prova vale um total de **10.0** pontos.

**Exercício 1**: Crie uma query que retorne o nome, profissão e empresa de todos os convidados que NÃO estão ativos na blocklist e cujo cachê seja menor que `700.00`.

**Obs**:

- Ordene de forma crescente pelo nome.
- Analise todos os campos da tabela `convidado` para entender o que é `blocklist`.
- `0` significa que o convidado não está na blocklist (Falso), `1` significa que está (Verdadeiro).

In [5]:
sql_ex01 = """
    SELECT nome, profissao, empresa
    FROM convidado
    WHERE blocklisted = 0
    AND cache < 700

    ORDER BY nome
"""

db(sql_ex01)

Executando query:
('Diego Fernandes', 'Barista', 'Coffee Lab')
('Dr. Sérgio Martins', 'Veterinário', 'Zoo de São Paulo')
('João Pedro', 'Influenciador Digital', 'YouTube Brasil')
('Letícia Gomes', 'Fashion Designer', 'São Paulo Fashion Week')
('Marcelo Tavares', 'Sommelier', 'Instituto do Vinho')
('Mônica Silva', 'Musicoterapeuta', 'Clínica Harmonia')
('Rafael Nunes', 'Personal Trainer', 'Academia Forma')
('Silvia Rocha', 'Terapeuta Holística', 'Centro Zen')


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [6]:
ia.sender(answer="sql_ex01", task="ai_md_25_2", question="ex01", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex01', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 2**: Crie uma query que retorne o nome do podcast, o título do episódio, a duração em minutos do episódio e o nome da categoria do podcast.

**Obs**: 
- Retorne apenas episódios que são públicos e que tenham duração maior que 70 minutos.
- Utilize a nomenclatura (`nome_podcast`, `titulo_episodio`, `duracao_minutos`, `categoria`).
- Ordene de forma decrescente pela duração em minutos.

In [9]:
sql_ex02 = """
    SELECT pod.nome AS nome_podcast, ep.titulo AS titulo_episodio, ep.duracao_minutos AS duracao_minutos, pod.categoria AS categoria
    FROM podcast AS pod
    INNER JOIN episodio AS ep ON ep.id_podcast = pod.id_podcast

    WHERE ep.publico = 1 AND duracao_minutos > 70

    ORDER BY duracao_minutos DESC
"""

db(sql_ex02)

Executando query:
('TechTalks Brasil', 'Quantum Computing: A Próxima Revolução', 102, 'Tecnologia')
('TechTalks Brasil', 'O Metaverso e o Futuro do Trabalho', 95, 'Tecnologia')
('TechTalks Brasil', 'Machine Learning na Prática', 93, 'Tecnologia')
('TechTalks Brasil', 'Startups Unicórnio: Cases de Sucesso', 92, 'Tecnologia')
('Negócios em Foco', 'ESG: Sustentabilidade como Estratégia', 91, 'Negócios')
('TechTalks Brasil', 'Ética em IA: Limites e Responsabilidades', 89, 'Tecnologia')
('TechTalks Brasil', 'Cibersegurança em Tempos de Home Office', 88, 'Tecnologia')
('TechTalks Brasil', 'FinTech: Democratizando Serviços Financeiros', 87, 'Tecnologia')
('TechTalks Brasil', 'O Futuro da Inteligência Artificial no Brasil', 85, 'Tecnologia')
('Negócios em Foco', 'Fusões e Aquisições no Brasil', 85, 'Negócios')
('Negócios em Foco', 'Liderança em Tempos de Crise', 82, 'Negócios')
('Mente Sana', 'Neuroplasticidade e Aprendizado', 79, 'Saúde')
('TechTalks Brasil', 'Blockchain além das Criptomoedas

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [17]:
ia.sender(answer="sql_ex02", task="ai_md_25_2", question="ex02", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex02', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 3**: A plataforma deseja analisar o engajamento dos hosts em podcasts. Crie uma query que retorne:

- O nome do host (coluna `nome_host`);
- O cachê por episódio do host (coluna `cache_por_episodio`);
- O número total de podcasts que o host apresenta (coluna `qtde_podcasts`);
- O número total de episódios que o host apresenta (coluna `qtde_episodios`);
- O valor total que seria pago se todos os episódios dos podcasts do host fossem gravados (coluna `valor_total_potencial`).

**Obs**:

- Cachê representa um valor monetário (dinheiro) pago ao host!
- Hosts sem podcasts (ou podcasts sem episódios) também devem ser retornados com valores zerados (onde se aplica).
- Para facilitar, iremos ignorar a data de gravação do episódio e as datas de início e fim da participação do host no podcast. Em todos os cálculos, considere todos os podcasts e episódios que o host apresenta.
- Ao calcular `qtde_episodios`, considere todos os episódios dos podcasts que o host apresenta.
- Para calcular o `valor_total_potencial`, multiplique o número de episódios de cada podcast pelo cachê do host. Considere que o host recebe por todos os episódios dos podcasts que apresenta.
- A função (papel) do host no podcast que apresenta está representada na tabela `podcast_host`.
- Um host pode ter mais múltiplos papeis no mesmo podcast, mas recebe apenas por um!
- Ordene de forma decrescente pelo `valor_total_potencial`. Em caso de empate, ordene de forma crescente pelo nome do host.

In [29]:
sql_ex03 = """
    SELECT h.nome AS nome_host, 
    h.cache_por_episodio AS cache_por_episodio,

    COUNT( DISTINCT pod.id_podcast)
    AS qtde_podcasts,

    COUNT( DISTINCT ep.id_episodio)
    AS qtde_episodios,

    COUNT(DISTINCT ep.id_episodio) * h.cache_por_episodio
    AS valor_total_potencial

    FROM podcast_host AS p_h
    LEFT JOIN podcast AS pod ON pod.id_podcast = p_h.id_podcast
    LEFT JOIN host AS h ON h.id_host = p_h.id_host
    LEFT JOIN episodio AS ep ON ep.id_podcast = pod.id_podcast

    GROUP BY h.id_host
"""

db(sql_ex03)

Executando query:
('Carlos Silva', Decimal('800.00'), 3, 16, Decimal('12800.00'))
('Ana Rodrigues', Decimal('750.00'), 1, 6, Decimal('4500.00'))
('Dr. Pedro Santos', Decimal('900.00'), 1, 5, Decimal('4500.00'))
('Felipe Martins', Decimal('650.00'), 3, 18, Decimal('11700.00'))
('Júlia Ferreira', Decimal('700.00'), 1, 4, Decimal('2800.00'))
('Camila Souza', Decimal('500.00'), 2, 0, Decimal('0.00'))
('Lucas Oliveira', Decimal('400.00'), 1, 3, Decimal('1200.00'))
('Beatriz Alves', Decimal('550.00'), 1, 3, Decimal('1650.00'))


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [32]:
ia.sender(answer="sql_ex03", task="ai_md_25_2", question="ex03", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex03', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 4**: A equipe de marketing deseja analisar o desempenho dos anúncios por anunciante.

Crie uma view `v_relatorio_anunciantes` que retorne:

- O nome da empresa anunciante (coluna `nome_empresa`);
- O segmento de mercado da empresa (coluna `segmento_mercado`);
- O número total de anúncios da empresa (coluna `qtde_anuncios`);
- O número total de vezes que os anúncios da empresa foram lidos em episódios (coluna `qtde_leituras`);
- A receita total gerada pelos anúncios da empresa (coluna `receita_total`);
- A receita média por menção gerada pelos anúncios da empresa (coluna `receita_media_mencao`).

**Obs**:
- Considere apenas anuncios ativos e com leituras (tabela `anuncio_episodio`) lidas completamente (atributo `lido_completo`).
- Considere apenas anunciantes que têm pelo menos 3 anúncios ativos.
- Ordene de forma decrescente pela receita total.
- A receita total é calculada considerando o valor efetivamente pago quando o anúncio foi veiculado no episódio (ver tabela `anuncio_episodio`).
- A receita média por menção é calculada considerando o valor efetivamente pago quando o anúncio foi veiculado no episódio (ver tabela `anuncio_episodio`).
- Utilize apenas uma query para criar a view.
- Não envie comando de `DROP` para o servidor.

In [ ]:
sql_ex04 = """
"""

db(sql_ex04)

Executando query:
0 linhas afetadas.


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [110]:
ia.sender(answer="sql_ex04", task="ai_md_25_2", question="ex04", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex04', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 5**: A plataforma XTREMO lançou um novo tipo de anúncio chamado **"Sponsored Content"** que deve ter duração de 60 segundos e preço base de 300.00.

Crie uma query para inserir este novo tipo de anúncio na base de dados.

**Obs**:

- Utilize o ID 5
- Utilize descrição `'Conteúdo patrocinado integrado ao episódio com narrativa natural'`.

In [ ]:
sql_ex05 = """
"""

# Descomente a linha abaixo para rodar localmente
db(sql_ex05)

Executando query:
1 linhas afetadas.


Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [58]:
ia.sender(answer="sql_ex05", task="ai_md_25_2", question="ex05", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex05', style=ButtonStyle()), Output()), _dom_classes=('widget…

**Exercício 6**: No diagrama do modelo relacional, observe a tabela `mensagem`.

Esta tabela foi criada para registrar as interações dos ouvintes relacionadas aos episódios (por exemplo, comentários no Youtube).

Perceba que:

- Falta uma chave extrangeira para a tabela `episodio` (um comentário deve estar relacionado a um episódio).
- Falta um atributo para armazenar o texto da mensagem.
- O atributo `data_envio` deve armazenar a data e hora da mensagem e não um texto.

Crie uma query para alterar a tabela `mensagem` e realizar as modificações necessárias.

**Obs**:

- Você deve alterar a tabela. Apagar e recriar não será considerada uma solução válida.
- O campo  `data_envio` deve ser do tipo `DATETIME`.
- Crie um atributo opcional `texto` do tipo `TEXT` para armazenar o texto da mensagem.
- Envie apenas uma query.

In [ ]:
sql_ex06 = """
"""

# Descomente a linha abaixo para rodar localmente
db(sql_ex06)

Executando query:


ProgrammingError: 1060 (42S21): Duplicate column name 'texto'

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [83]:
ia.sender(answer="sql_ex06", task="ai_md_25_2", question="ex06", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex06', style=ButtonStyle()), Output()), _dom_classes=('widget…

### Conferindo as notas!

In [84]:
ia.grades(task="ai_md_25_2")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | ai_md_25_2  | ex01        |      2 |     10 |                10 |                 0 |
|  1 | ai_md_25_2  | ex02        |      3 |     10 |                10 |                 0 |
|  2 | ai_md_25_2  | ex03        |      3 |      0 |                 0 |                 0 |
|  3 | ai_md_25_2  | ex04        |      3 |      0 |                 0 |                 0 |
|  4 | ai_md_25_2  | ex05        |      2 |     10 |                10 |                 0 |
|  5 | ai_md_25_2  | ex06        |      3 |     10 |                10 |                 0 |

## Parte 2 - Sistema de Gestão de Locadora de Equipamentos

A **EquipaLock** é uma empresa especializada no aluguel de de equipamentos para os setores de construção civil, eventos, jardinagem e projetos industriais, atendendo tanto clientes pessoa física quanto jurídica.

Você foi contratado pela **EquipaLock** para desenvolver um sistema que permita gerenciar a locação de equipamentos profissionais (ferramentas, máquinas e equipamentos diversos) para clientes.

O objetivo é controlar o inventário de equipamentos, registrar locações, calcular valores e acompanhar devoluções.

Em reuniões com a equipe da **EquipaLock**, foram levantadas as seguintes informações sobre o domínio de negócio:

- **Cadastro de Categorias de Equipamentos**: Os equipamentos são organizados em categorias para facilitar a busca e gestão. Cada categoria deve ter um nome único e uma descrição longa. Todos os atributos são obrigatórios.

- **Cadastro de Equipamentos**: O sistema deve permitir o cadastro de equipamentos disponíveis para locação, incluindo nome do equipamento, descrição, número de série (único), categoria, peso, valor da diária de locação (valor monetário) e status atual (disponível, locado, em manutenção). Cada equipamento pertence a uma única categoria de equipamentos. É opcional fornecer o peso do equipamento, todos os outros campos são obrigatórios.

- **Cadastro de Clientes**: É necessário cadastrar os clientes que podem alugar equipamentos. As informações incluem: nome completo, CPF ou CNPJ (único), telefone, e-mail e endereço completo (rua, número, cidade, estado). Exceto o CPF/CNPJ, todos os campos são obrigatórios.

- **Registro de Locações**: O sistema deve registrar cada locação realizada. Uma locação representa o aluguel de um ou mais equipamentos por um cliente. Para cada locação, deve-se armazenar: o cliente que está locando, a data de início da locação, a data prevista de devolução e a data efetiva de devolução (opcional, preenchida quando o equipamento é devolvido). Cada locação pode incluir múltiplos equipamentos, e para cada equipamento locado deve-se registrar o valor da diária contratada, uma vez que o valor da diária no cadastro de equipamentos irá variar ao longo do tempo.

**Observações**:

- A empresa deseja que todas as tabelas possuam um identificador único (chave primária) do tipo inteiro.
- Em uma locação, considere que todos os equipamentos são devolvidos na mesma data (data efetiva de devolução).
- Evite redundância de dados e garanta a integridade referencial entre as tabelas.
- Pense nos tipos de dados adequados para cada atributo (por exemplo, datas, valores monetários, textos) e seus tamanhos.

**Tarefa:**

Com base nas informações fornecidas pela equipe da **EquipaLock**, elabore um **diagrama do modelo relacional (no Workbench)** que represente as entidades, atributos e relacionamentos necessários para implementar o sistema descrito. Certifique-se de identificar chaves primárias, chaves estrangeiras e quaisquer restrições de integridade relevantes.

**Dica**:

- Evite cruzar linhas de relacionamentos no diagrama.
- Evite deixar atributos ocultos ou com seus nomes cortados.

**Exercício 7**: Desenhe o diagrama do modelo relacional com uma solução para o problema. Não esqueça de indicar claramente os atributos (obrigatórios e opcionais), tipos, chaves primárias, chaves estrangeiras, e a cardinalidade dos relacionamentos.

**Obs:** Salve a imagem do diagrama na pasta `img`. Edite na resposta o caminho para a imagem!

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">
    
<img src="img/Captura de tela 2025-10-01 120307.png">

</div>

**Opcional**: caso julgue necessário, utilize o bloco abaixo para justificar decisões tomadas durante o desenho do diagrama.

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">
    
Atraves da tabela "locacao" conseguimos ver qual cliente está alugando o equipamento 'x' e o valor total deve ser alterado todo dia

</div>

**Opcional**: caso julgue necessário, utilize o bloco abaixo para relatar algo sobre a prova como um todo

<div style="background-color: rgba(71, 85, 155, 0.2); padding: 10px; border-radius: 5px; border-left: 4px solid #475569; border: 1px solid rgba(71, 85, 105, 0.3);">
    
Seu texto AQUI!

</div>

## Entrega!

É hora de entregar. Faça `zip` da pasta de prova (zip, não rar) e envie no Blackboard.